# Simple RAG-LLM Action Planning

This notebook loads predictions, searches knowledge base using contributing features, and generates action plans using LLM.

In [12]:
# All Imports
import os
import json
import re
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from openai import OpenAI

load_dotenv()

True

In [13]:
# 1. Setup and Load Data
# Paths
rag_predictions_dir = Path("RAG_docs/predictions")
rag_knowledge_dir = Path("RAG_docs/knowledge")
results_action_dir = Path("RAG_docs/action_plans")
os.makedirs(results_action_dir, exist_ok=True)

# Functional loading functions
def load_predictions(predictions_dir: Path) -> List[Dict[str, Any]]:
    """Load all prediction files from the specified directory.
    
    Args:
        predictions_dir: Path to directory containing prediction JSON files
    
    Returns:
        List of prediction dictionaries with added _source_file metadata
    
    Raises:
        FileNotFoundError: If no JSON files found in directory
    """
    prediction_files = list(predictions_dir.glob("*.json"))
    if not prediction_files:
        raise FileNotFoundError(f"No JSON files found in {predictions_dir}")
    
    print(f"Found {len(prediction_files)} prediction file(s) in {predictions_dir}")
    
    predictions_data = []
    for prediction_file in prediction_files:
        print(f"  Loading: {prediction_file.name}")
        with open(prediction_file, "r", encoding="utf-8") as f:
            file_data = json.load(f)
        
        # Handle single object or array
        if isinstance(file_data, dict):
            file_data = [file_data]
        elif isinstance(file_data, list):
            pass  # Already a list
        else:
            print(f"    Warning: Unexpected data type in {prediction_file.name}, skipping")
            continue
        
        # Add source file info to each prediction
        for sample in file_data:
            if isinstance(sample, dict):
                sample["_source_file"] = prediction_file.name
        
        predictions_data.extend(file_data)
        print(f"    Loaded {len(file_data)} prediction(s) from {prediction_file.name}")
    
    print(f"\nTotal: Loaded {len(predictions_data)} prediction(s) from {len(prediction_files)} file(s)")
    return predictions_data

# Knowledge base file converters
# All converters follow the same structure: accept a file path and return list of documents
# These must be defined BEFORE load_knowledge_base() which uses them
def load_ddos_dataset(file_path: Path) -> List[Dict[str, Any]]:
    """Load DDoS synthetic dataset from JSON file.
    
    Args:
        file_path: Path to ddos-synthetic-dataset.json file
    
    Returns:
        List of document dictionaries (already in {title, text} format)
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # DDoS dataset is already in document format (list of {title, text})
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "title" in data and "text" in data:
        return [data]
    else:
        # Try to convert if it's a different structure
        return []

def load_agentic_features(file_path: Path) -> List[Dict[str, Any]]:
    """Load and convert agentic_features_actions.json to document format.
    
    Args:
        file_path: Path to agentic_features_actions.json file
    
    Returns:
        List of document dictionaries with {title, text} format
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    documents = []
    
    # Document 1: Overview and metadata
    if "description" in data:
        text_parts = [data["description"]]
        if "metadata" in data:
            metadata = data["metadata"]
            if "mitigation_techniques" in metadata:
                text_parts.append(f"\n\nAvailable mitigation techniques: {', '.join(metadata['mitigation_techniques'])}")
            if "notes" in metadata:
                text_parts.append("\n\nNotes:")
                for note in metadata["notes"]:
                    text_parts.append(f"  - {note}")
        documents.append({
            "title": "Agentic Features and Actions - Overview",
            "text": "\n".join(text_parts)
        })
    
    # Document 2: RAN features and actions
    if "RAN" in data:
        ran = data["RAN"]
        features = ran.get("features", [])
        actions = ran.get("actions", [])
        text_parts = [
            "RAN (Radio Access Network) network tier features and available mitigation actions.",
            f"\n\nAvailable Features ({len(features)} total):",
            ", ".join(features[:50])
        ]
        if len(features) > 50:
            text_parts.append(f"\n... and {len(features) - 50} more features")
        text_parts.append(f"\n\nAvailable Actions: {', '.join(actions)}")
        documents.append({
            "title": "RAN (Radio Access Network) Features and Actions",
            "text": "\n".join(text_parts)
        })
    
    # Document 3: Edge features and actions
    if "Edge" in data:
        edge = data["Edge"]
        features = edge.get("features", [])
        actions = edge.get("actions", [])
        text_parts = [
            "Edge network tier features and available mitigation actions.",
            f"\n\nAvailable Features ({len(features)} total):",
            ", ".join(features[:50])
        ]
        if len(features) > 50:
            text_parts.append(f"\n... and {len(features) - 50} more features")
        text_parts.append(f"\n\nAvailable Actions: {', '.join(actions)}")
        documents.append({
            "title": "Edge Network Features and Actions",
            "text": "\n".join(text_parts)
        })
    
    # Document 4: Core features and actions
    if "Core" in data:
        core = data["Core"]
        features = core.get("features", [])
        actions = core.get("actions", [])
        text_parts = [
            "Core network tier features and available mitigation actions.",
            f"\n\nAvailable Features ({len(features)} total):",
            ", ".join(features[:50])
        ]
        if len(features) > 50:
            text_parts.append(f"\n... and {len(features) - 50} more features")
        text_parts.append(f"\n\nAvailable Actions: {', '.join(actions)}")
        documents.append({
            "title": "Core Network Features and Actions",
            "text": "\n".join(text_parts)
        })
    
    return documents

def load_attack_options(file_path: Path) -> List[Dict[str, Any]]:
    """Load and convert attack_options.json to document format.
    
    Args:
        file_path: Path to attack_options.json file
    
    Returns:
        List of document dictionaries with {title, text} format
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    documents = []
    
    # Create a comprehensive document with all attack types and their recommended actions
    if "ATTACK_ACTIONS" in data:
        text_parts = []
        
        # Add description if available
        if "description" in data:
            text_parts.append(data["description"])
        
        text_parts.append("\n\nRecommended mitigation actions by attack type:\n")
        
        # Add each attack type with its recommended actions
        for attack_type, actions in data["ATTACK_ACTIONS"].items():
            text_parts.append(f"\n{attack_type}:")
            text_parts.append(f"  Recommended actions: {', '.join(actions)}")
        
        documents.append({
            "title": "Attack Classification and Mitigation Actions",
            "text": "\n".join(text_parts)
        })
    
    return documents

def load_knowledge_base(knowledge_dir: Path) -> List[Dict[str, Any]]:
    """Load all knowledge base files from the knowledge directory.
    
    Automatically discovers and loads all JSON files in the directory, routing them
    to appropriate converters based on filename patterns.
    
    Args:
        knowledge_dir: Path to directory containing knowledge base JSON files
    
    Returns:
        List of knowledge base documents
    """
    knowledge_base = []
    
    # Get all JSON files in the knowledge directory
    knowledge_files = list(knowledge_dir.glob("*.json"))
    if not knowledge_files:
        raise FileNotFoundError(f"No JSON files found in {knowledge_dir}")
    
    print(f"\nLoading knowledge base files from {knowledge_dir}:")
    
    # File routing: map filename patterns to converter functions
    file_routes = {
        "agentic_features_actions.json": load_agentic_features,
        "attack_options.json": load_attack_options,
        "ddos-synthetic-dataset.json": load_ddos_dataset,
    }
    
    # Load each file using appropriate converter
    for knowledge_file in sorted(knowledge_files):
        filename = knowledge_file.name
        
        if filename in file_routes:
            # Use specific converter for known file types
            converter = file_routes[filename]
            docs = converter(knowledge_file)
            knowledge_base.extend(docs)
            print(f"  ✓ Loaded {len(docs)} document(s) from {filename}")
        else:
            # Default: try to load as standard document format (list of {title, text})
            try:
                with open(knowledge_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                
                # Handle list of documents
                if isinstance(data, list):
                    knowledge_base.extend(data)
                    print(f"  ✓ Loaded {len(data)} document(s) from {filename} (standard format)")
                # Handle single document
                elif isinstance(data, dict) and "title" in data and "text" in data:
                    knowledge_base.append(data)
                    print(f"  ✓ Loaded 1 document from {filename} (standard format)")
                else:
                    print(f"  ⚠ Skipped {filename} (unknown format)")
            except Exception as e:
                print(f"  ✗ Error loading {filename}: {e}")
    
    print(f"\nTotal: {len(knowledge_base)} documents in knowledge base")
    return knowledge_base

# Load predictions and knowledge base using functional approach
predictions_data = load_predictions(rag_predictions_dir)
knowledge_base = load_knowledge_base(rag_knowledge_dir)


Found 1 prediction file(s) in RAG_docs\predictions
  Loading: sample_predictions_detailed_20260207_111609.json
    Loaded 2 prediction(s) from sample_predictions_detailed_20260207_111609.json

Total: Loaded 2 prediction(s) from 1 file(s)

Loading knowledge base files from RAG_docs\knowledge:
  ✓ Loaded 4 document(s) from agentic_features_actions.json
  ✓ Loaded 1 document(s) from attack_options.json
  ✓ Loaded 121 document(s) from ddos-synthetic-dataset.json

Total: 126 documents in knowledge base


In [14]:
# 2. Create Vector Store (Offline - No API Key Required)
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Process knowledge base into documents
rag_documents = []
for idx, item in enumerate(knowledge_base):
    title = item.get("title", "")
    text = item.get("text", "")
    doc_text = f"Title: {title}\n\nContent: {text}"
    
    chunks = text_splitter.split_text(doc_text)
    for chunk_idx, chunk in enumerate(chunks):
        rag_documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "text": text, "item_index": idx, "chunk_index": chunk_idx}
        ))

print(f"Created {len(rag_documents)} document chunks")

# Create vector store using local embeddings (offline)
print("Creating vector embeddings using local model (this may take a few minutes)...")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(rag_documents, embeddings)
print("✓ Vector store created (offline, no API key required)")


Created 457 document chunks
Creating vector embeddings using local model (this may take a few minutes)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Vector store created (offline, no API key required)


In [15]:
# 4a. Extract Party Features (Reusable Function)
def build_features_context(sample: Dict[str, Any], top_n: int = 10) -> str:
    """Build formatted context string for contributing features by party from prediction file.
    
    Extracts party list directly from party_contributions and features from feature_contributions,
    then groups them by network tier (RAN, Edge, Core).
    
    Args:
        sample: Prediction sample dictionary with SHAP explanations
        top_n: Number of top features to include per party (default: 10)
    
    Returns:
        Formatted string with contributing features grouped by network tier, or empty string if no features
    """
    shap_expl = sample.get("shap_explanation", {}) or {}
    
    # Get party list directly from party_contributions
    party_contributions = shap_expl.get("party_contributions", {}) or {}
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    
    if not party_contributions or not feat_contribs:
        return ""
    
    # Group features by network tier
    tier_features: Dict[str, List[Dict[str, Any]]] = {"RAN": [], "Edge": [], "Core": []}
    
    # Get party-to-tier mapping to avoid duplicate logic
    party_to_tier = get_party_to_tier_mapping(sample)
    
    # Iterate through parties from party_contributions
    for party_name in party_contributions.keys():
        if party_name not in feat_contribs:
            continue
        
        feats = feat_contribs[party_name]
        if not isinstance(feats, dict):
            continue
        
        # Get network tier from mapping
        network_tier = party_to_tier.get(party_name, "")
        if not network_tier:
            continue  # Skip if no tier match
        
        # Extract features for this party
        party_feat_list = []
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            abs_val = float(v.get("abs_shap_value", 0.0) or 0.0)
            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)
            if abs_val > 0:
                party_feat_list.append({
                    "name": feat_name,
                    "abs_shap_value": abs_val,
                    "pct_contribution": pct_contrib,
                    "party": party_name
                })
        
        # Sort by absolute SHAP value and add to tier
        party_feat_list.sort(key=lambda x: x["abs_shap_value"], reverse=True)
        tier_features[network_tier].extend(party_feat_list[:top_n])
    
    # Build context string
    if not any(tier_features.values()):
        return ""
    
    features_context = "\n\nContributing Features (by network tier, contribution > 0):\n"
    
    for tier in ["RAN", "Edge", "Core"]:
        if tier_features[tier]:
            # Deduplicate features by name (keep highest contribution)
            seen = {}
            for feat in tier_features[tier]:
                name = feat["name"]
                if name not in seen or feat["abs_shap_value"] > seen[name]["abs_shap_value"]:
                    seen[name] = feat
            
            # Sort and display
            unique_feats = sorted(seen.values(), key=lambda x: x["abs_shap_value"], reverse=True)
            features_context += f"\n{tier} Network Tier:\n"
            for feat in unique_feats[:top_n]:
                features_context += f"  - {feat['name']}: {feat['pct_contribution']:.2%} contribution (abs_shap: {feat['abs_shap_value']:.4f}) [Party: {feat['party']}]\n"
    
    return features_context

In [16]:
# 4. Smart RAG Query Generation (Feature-name -> Human Signal Rules)
# Feature-name -> human signal rules
SIGNAL_RULES: List[Tuple[re.Pattern, str]] = [
    # DNS / UDP
    (re.compile(r"dns", re.I), "abnormal DNS activity (possible amplification/fan-in)"),
    (re.compile(r"\budps?\b", re.I), "elevated UDP traffic patterns"),
    (re.compile(r"dns_port(_src)?_count", re.I), "DNS source fan-in / query-source spike"),

    # TCP flags
    (re.compile(r"\brst(_packets)?\b", re.I), "elevated TCP RST responses (reset bursts)"),
    (re.compile(r"\bsyn(_packets)?\b", re.I), "SYN surge behavior (flood-like pattern)"),
    (re.compile(r"\back(_packets)?\b", re.I), "abnormal ACK behavior"),
    (re.compile(r"\bpsh(_packets)?\b", re.I), "PSH-flag anomalies"),

    # Timing
    (re.compile(r"\bpiat\b", re.I), "inter-arrival timing irregularities"),
    (re.compile(r"duration", re.I), "flow duration anomalies"),

    # Volume / size
    (re.compile(r"bytes", re.I), "unusual byte volume distribution"),
    (re.compile(r"packet_count|\bpackets\b", re.I), "packet-rate / packet-count anomalies"),
    (re.compile(r"_mean_ps\b|packet_size", re.I), "packet-size shift (mean/variance change)"),

    # Ports / scan-like
    (re.compile(r"unique_ports|vul_ports|http_ports", re.I), "suspicious port/service targeting patterns"),
]

def extract_top_features(sample: Dict[str, Any], top_n: int = 7) -> List[str]:
    """Top feature names by abs_shap_value across all parties (non-zero)."""
    feat_contribs = (sample.get("shap_explanation", {}) or {}).get("feature_contributions", {}) or {}

    scored: List[Tuple[float, str]] = []
    for _, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            abs_val = float(v.get("abs_shap_value", 0.0) or 0.0)
            if abs_val > 0:
                scored.append((abs_val, feat_name))

    scored.sort(reverse=True, key=lambda x: x[0])

    seen = set()
    top_feats = []
    for _, f in scored:
        if f not in seen:
            seen.add(f)
            top_feats.append(f)
        if len(top_feats) >= top_n:
            break
    return top_feats

def features_to_signals(feature_names: List[str], max_signals: int = 3) -> List[str]:
    """Map feature names -> deduped human signals."""
    signals: List[str] = []
    for f in feature_names:
        for pattern, signal in SIGNAL_RULES:
            if pattern.search(f):
                if signal not in signals:
                    signals.append(signal)
                break  # only first match per feature
        if len(signals) >= max_signals:
            break
    return signals

def get_party_to_tier_mapping(sample: Dict[str, Any]) -> Dict[str, str]:
    """Get mapping of party names to network tiers directly from sample data.
    
    Extracts party names from sample's party_contributions and maps each to its network tier.
    Party names from sample: "RAN_device_party1", "Edge_device_party2", "Core_device_party3"
    
    Args:
        sample: Prediction sample dictionary with SHAP explanations
    
    Returns:
        Dictionary mapping party names to network tier names (e.g., {"RAN_device_party1": "RAN", ...})
    """
    shap_expl = sample.get("shap_explanation", {}) or {}
    party_contributions = shap_expl.get("party_contributions", {}) or {}
    
    party_to_tier = {}
    for party_name in party_contributions.keys():
        # Extract network tier directly from party name
        party_lower = party_name.lower()
        if party_lower.startswith("ran"):
            tier = "RAN"
        elif party_lower.startswith("edge"):
            tier = "Edge"
        elif party_lower.startswith("core"):
            tier = "Core"
        else:
            continue  # Skip if no tier match
        
        party_to_tier[party_name] = tier
    
    return party_to_tier

def extract_top_features_by_party(sample: Dict[str, Any], top_n: int = 5) -> Dict[str, List[Tuple[str, float, float]]]:
    """Extract top features from each party with their contributions.
    
    Args:
        sample: Prediction sample dictionary
        top_n: Number of top features per party
    
    Returns:
        Dictionary mapping party names to list of (feature_name, abs_shap_value, pct_contribution) tuples
    """
    feat_contribs = (sample.get("shap_explanation", {}) or {}).get("feature_contributions", {}) or {}
    
    party_features: Dict[str, List[Tuple[str, float, float]]] = {}
    
    for party_name, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        
        scored: List[Tuple[float, str, float]] = []
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            abs_val = float(v.get("abs_shap_value", 0.0) or 0.0)
            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)
            if abs_val > 0:
                scored.append((abs_val, feat_name, pct_contrib))
        
        # Sort by absolute SHAP value (descending)
        scored.sort(reverse=True, key=lambda x: x[0])
        
        # Store top N features for this party
        party_features[party_name] = [(name, abs_val, pct) for abs_val, name, pct in scored[:top_n]]
    
    return party_features

def get_dominant_party_info(sample: Dict[str, Any]) -> Tuple[str, str, float]:
    """Get dominant party information using party-to-tier mapping.
    
    Args:
        sample: Prediction sample dictionary
    
    Returns:
        Tuple of (party_name, network_tier, contribution_percentage)
    """
    shap_expl = sample.get("shap_explanation", {}) or {}
    dominant_agent = shap_expl.get("dominant_agent", "")
    dominant_pct = shap_expl.get("dominant_contribution_pct", 0.0) or shap_expl.get("party_contributions_pct", {}).get(dominant_agent, 0.0)
    
    if isinstance(dominant_pct, dict):
        dominant_pct = dominant_pct.get(dominant_agent, 0.0)
    
    # Get network tier using party-to-tier mapping (removes duplicate logic)
    party_to_tier = get_party_to_tier_mapping(sample)
    network_tier = party_to_tier.get(dominant_agent, "") if dominant_agent else ""
    
    return (dominant_agent, network_tier, float(dominant_pct) * 100.0)

def build_dynamic_rag_query(sample: Dict[str, Any],
                            intent: str = "What mitigation actions",
                            max_signals: int = 3,
                            include_network_tier: bool = True) -> str:
    """Build an action-oriented RAG query based on agentic feature contributions.
    
    This function creates queries that specifically request actionable mitigation steps
    based on contributing features from each network tier (RAN, Edge, Core).
    
    This function:
    1. Extracts top contributing features from each party (RAN, Edge, Core)
    2. Maps parties to network tiers and their contribution percentages
    3. Identifies dominant party and network tier
    4. Builds an action-focused query optimized to retrieve relevant mitigation actions
    
    Args:
        sample: Prediction sample dictionary with SHAP explanations
        intent: Query intent (default: "What mitigation actions")
        max_signals: Maximum number of signal descriptions to include
        include_network_tier: Whether to include network tier context in query
    
    Returns:
        Action-oriented RAG query string optimized for retrieving mitigation actions
    """
    label = sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"
    
    # Get dominant party and network tier
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample)
    
    # Extract top features by party with their contributions
    party_features = extract_top_features_by_party(sample, top_n=5)
    party_to_tier = get_party_to_tier_mapping(sample)
    
    # Get party contributions for action prioritization
    shap_expl = sample.get("shap_explanation", {}) or {}
    party_contributions_pct = shap_expl.get("party_contributions_pct", {}) or {}
    
    # Get top features across all parties for signal extraction
    top_feats = extract_top_features(sample, top_n=10)
    signals = features_to_signals(top_feats, max_signals=max_signals)
    
    # Build action-oriented query components
    query_parts = []
    
    # 1. Action-focused intent with attack type
    query_parts.append(f"{intent} are required for {label} attack")
    
    # 2. Network tier context with contribution (critical for action assignment)
    if include_network_tier and dominant_tier:
        query_parts.append(f"detected in {dominant_tier} network tier")
        if dominant_pct > 0:
            query_parts.append(f"({dominant_tier} contributing {dominant_pct:.1f}%)")
    
    # 3. Feature-based action triggers (most important for agentic actions)
    if signals:
        if len(signals) == 1:
            signals_text = signals[0]
        elif len(signals) == 2:
            signals_text = f"{signals[0]} and {signals[1]}"
        else:
            signals_text = ", ".join(signals[:-1]) + f", and {signals[-1]}"
        query_parts.append(f"with {signals_text}")
    
    # 4. Network tier-specific high-contribution features (for targeted actions)
    tier_action_features = []
    all_contributing_features = []  # Collect all contributing features for RAG search
    
    for party_name, features in party_features.items():
        if not features:
            continue
        
        tier = party_to_tier.get(party_name, "")
        if not tier:
            continue
        
        # Get contribution percentage for this party
        party_pct = float(party_contributions_pct.get(party_name, 0.0) or 0.0)
        
        # Include features from parties with significant contribution (>10%)
        if party_pct > 10.0:
            # Get top 3 features with highest contribution (for better RAG matching)
            top_feat_names = [name for name, _, pct in features[:3] if pct > 0.05]
            if top_feat_names:
                tier_action_features.append((tier, top_feat_names[0] if top_feat_names else None))
                # Add all contributing features to list for RAG query
                all_contributing_features.extend(top_feat_names)
    
    # 5. Add specific high-contribution features for action targeting
    if tier_action_features:
        # Focus on dominant tier features first
        dominant_tier_feats = [feat for tier, feat in tier_action_features if tier == dominant_tier and feat]
        if dominant_tier_feats:
            query_parts.append(f"requiring actions for features: {dominant_tier_feats[0]}")
    
    # 5b. Add ALL contributing feature names with >0% contribution to query for better RAG matching
    # Extract all features with >0% contribution from all parties
    all_features_with_contribution = []
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    
    for party_name, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)
            # Include all features with >0% contribution
            if pct_contrib > 0.0:
                all_features_with_contribution.append(feat_name)
    
    # Add unique contributing features to query (all features with >0% contribution)
    if all_features_with_contribution:
        unique_features = list(dict.fromkeys(all_features_with_contribution))  # All unique features with >0%
        if unique_features:
            # Limit to reasonable number for query (max 20 features to avoid query being too long)
            feature_names_text = ", ".join(unique_features[:20])
            query_parts.append(f"with contributing features (>0%): {feature_names_text}")
            if len(unique_features) > 20:
                query_parts.append(f"and {len(unique_features) - 20} more contributing features")
    
    # 6. Action requirements based on network tier contributions
    action_requirements = []
    for party_name, contribution_pct in party_contributions_pct.items():
        if contribution_pct and float(contribution_pct) > 15.0:  # Significant contributors
            tier = party_to_tier.get(party_name, "")
            if tier:
                action_requirements.append(f"{tier} tier actions")
    
    if action_requirements:
        if len(action_requirements) == 1:
            query_parts.append(f"requiring {action_requirements[0]}")
        else:
            query_parts.append(f"requiring actions from: {', '.join(action_requirements)}")
    
    # Combine into final action-oriented query
    if len(query_parts) > 1:
        query = " ".join(query_parts) + " based on agentic feature contributions and network tier analysis."
    else:
        query = f"{intent} are required for {label} attack based on agentic feature contributions and provide specific mitigation actions for each network tier."
    
    return query

def get_actions_for_features(sample: Dict[str, Any], agentic_features_file: Path = None) -> Dict[str, Any]:
    """Retrieve actions from agentic_features_actions.json based on contributing features in sample data.
    
    This function directly maps features from sample data to available actions in the knowledge base,
    providing a deterministic feature-to-action mapping alongside RAG retrieval.
    
    Args:
        sample: Prediction sample dictionary with SHAP explanations
        agentic_features_file: Path to agentic_features_actions.json (default: uses rag_knowledge_dir)
    
    Returns:
        Dictionary with:
        - tier_actions: Dict mapping network tier to list of available actions
        - feature_matches: Dict mapping network tier to list of matching features
        - matched_features_count: Dict mapping network tier to count of matching features
        - all_actions: List of all unique actions across all tiers
    """
    # Get file path
    if agentic_features_file is None:
        agentic_features_file = rag_knowledge_dir / "agentic_features_actions.json"
    
    if not agentic_features_file.exists():
        return {
            "tier_actions": {},
            "feature_matches": {},
            "matched_features_count": {},
            "all_actions": [],
            "error": f"File not found: {agentic_features_file}"
        }
    
    # Load agentic features file
    with open(agentic_features_file, "r", encoding="utf-8") as f:
        agentic_data = json.load(f)
    
    # Extract contributing features from sample
    shap_expl = sample.get("shap_explanation", {}) or {}
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    party_contributions_pct = shap_expl.get("party_contributions_pct", {}) or {}
    
    # Get party-to-tier mapping
    party_to_tier = get_party_to_tier_mapping(sample)
    
    # Collect all contributing features (>0%) grouped by tier
    tier_features: Dict[str, List[str]] = {"RAN": [], "Edge": [], "Core": []}
    
    for party_name, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        
        tier = party_to_tier.get(party_name, "")
        if not tier or tier not in tier_features:
            continue
        
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)
            if pct_contrib > 0.0:
                tier_features[tier].append(feat_name)
    
    # Get actions for each tier based on feature matches
    tier_actions = {}
    feature_matches = {}
    matched_features_count = {}
    all_actions_set = set()
    
    for tier in ["RAN", "Edge", "Core"]:
        if tier not in agentic_data:
            continue
        
        tier_data = agentic_data[tier]
        tier_feature_list = set(tier_data.get("features", []))
        tier_action_list = tier_data.get("actions", [])
        
        # Find matching features (features in sample that exist in tier's feature list)
        sample_features = set(tier_features.get(tier, []))
        matching_features = list(sample_features.intersection(tier_feature_list))
        
        # Store results
        feature_matches[tier] = matching_features
        matched_features_count[tier] = len(matching_features)
        
        # If there are matching features, include actions for this tier
        if matching_features:
            tier_actions[tier] = tier_action_list
            all_actions_set.update(tier_action_list)
    
    return {
        "tier_actions": tier_actions,  # {tier: [actions]}
        "feature_matches": feature_matches,  # {tier: [matching_feature_names]}
        "matched_features_count": matched_features_count,  # {tier: count}
        "all_actions": list(all_actions_set),  # All unique actions
        "party_contributions": {
            party: float(pct) for party, pct in party_contributions_pct.items() if pct and float(pct) > 0
        }
    }

def get_feature_action_context(sample: Dict[str, Any], agentic_features_file: Path = None) -> str:
    """Get formatted feature-to-action context string for a sample.
    
    This is a convenience function that returns the formatted context string
    showing which actions are available for each network tier based on matching features.
    
    Args:
        sample: Prediction sample dictionary with SHAP explanations
        agentic_features_file: Path to agentic_features_actions.json (default: uses rag_knowledge_dir)
    
    Returns:
        Formatted string with feature-to-action mapping by network tier
    """
    feature_action_mapping = get_actions_for_features(sample, agentic_features_file)
    
    feature_action_context = ""
    if feature_action_mapping.get("tier_actions"):
        feature_action_context = "\nDirect Feature-to-Action Mapping (from agentic_features_actions.json):\n"
        for tier, actions in feature_action_mapping["tier_actions"].items():
            matched_count = feature_action_mapping["matched_features_count"].get(tier, 0)
            matching_features = feature_action_mapping["feature_matches"].get(tier, [])
            
            feature_action_context += f"\n{tier} Network Tier:\n"
            feature_action_context += f"  - Matching features: {matched_count} features with >0% contribution\n"
            if matching_features:
                # Show first 10 matching features as examples
                feature_examples = ", ".join(matching_features[:10])
                if len(matching_features) > 10:
                    feature_examples += f", ... ({len(matching_features) - 10} more)"
                feature_action_context += f"  - Example matching features: {feature_examples}\n"
            feature_action_context += f"  - Available actions: {', '.join(actions)}\n"
        
        all_actions = feature_action_mapping.get('all_actions', [])
        if all_actions:
            feature_action_context += f"\nAll available actions across tiers: {', '.join(all_actions)}\n"
    else:
        feature_action_context = "\nNo matching features found in agentic_features_actions.json\n"
    
    return feature_action_context

In [17]:
# 5. Create LLM Prompt with Party Evidence and RAG Context
# Updated create_prompt function for dynamic party system
# Replace the existing create_prompt function in cell 5 of RAG_LLM_action_plan.ipynb

def create_prompt(sample_data, rag_results):
    """Create prompt with prediction details, party evidence, contributing features, and RAG context (top 3 docs).
    
    Uses dynamic party system with network tiers (RAN, Edge, Core) and extracts party contributions
    dynamically from the sample data.
    """
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    # Extract party contributions dynamically
    shap_explanation = sample_data.get("shap_explanation", {}) or {}
    party_contributions_pct = shap_explanation.get("party_contributions_pct", {}) or {}
    party_contributions = shap_explanation.get("party_contributions", {}) or {}
    
    # Get dominant party information using dynamic system
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample_data)
    
    # Get party-to-tier mapping
    party_to_tier = get_party_to_tier_mapping(sample_data)
    
    # Build party evidence summary dynamically
    party_evidence_lines = []
    for party_name, contribution_pct in party_contributions_pct.items():
        if contribution_pct and float(contribution_pct) > 0:
            tier = party_to_tier.get(party_name, "Unknown")
            party_evidence_lines.append(f"- {party_name} ({tier} tier): {float(contribution_pct):.2%}")
    
    # If no dynamic parties found, fall back to empty or default
    if not party_evidence_lines:
        party_evidence_lines.append("- No party contributions found")
    
    party_evidence_summary = "\n".join(party_evidence_lines)
    
    # Build contributing features context using reusable function (works with dynamic parties)
    features_context = build_features_context(sample_data, top_n=10)
    
    # Get feature-to-action context using helper function
    feature_action_context = get_feature_action_context(sample_data)
    
    # Use top 3 RAG results
    top_3_results = rag_results[:3]
    
    # Build RAG context
    rag_context = ""
    if top_3_results:
        rag_context = "\n\nKnowledge Base Context:\n"
        for idx, result in enumerate(top_3_results, 1):
            rag_context += f"\n[{idx}] {result['title']}\n"
            rag_context += f"{result['text'][:1000]}\n"
    else:
        rag_context = "\n\nKnowledge Base: No relevant documents found."
    
    # Build network tier context for prompt
    network_tier_info = ""
    if dominant_tier:
        network_tier_info = f"\n- Dominant network tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)"
        if dominant_party:
            network_tier_info += f"\n- Dominant party: {dominant_party}"
    prompt = f"""You are a cybersecurity expert analyzing attack predictions and generating action plans.
        The goal is to support agent decisions on WHICH evidence party should trigger actions
        and WHAT actions are appropriate based on domain knowledge.

        Prediction summary:
        - sample_id: {sample_id}
        - predicted_label: {predicted_label}
        - confidence: {confidence:.1%}

        Explainability (party-level evidence by network tier):
        {party_evidence_summary}
        {network_tier_info}
        {features_context}
        {feature_action_context}

        {rag_context}

        Task:
        Based on the prediction details and knowledge base content above, provide an action plan that:
        1) Understands how {predicted_label} manifests in each network tier (RAN, Edge, Core)
        and associated evidence types (volume/rate, packet size, timing/direction).
        2) Decides which evidence party (by network tier) should trigger an agent action first.
        3) Determines what agent actions are appropriate for each evidence party and network tier
        (e.g., rate limiting, filtering, flow inspection, escalation).
        4) Validates whether non-dominant parties provide supporting or contradictory signals.
        5) Supports confidence-aware decisions (given confidence = {confidence:.1%}).
        6) CRITICAL: Each action MUST specify which party (use actual party name from party_contributions) will execute it.

        Output:
        Return a JSON object with the following structure:
        {{
        "threat_level": "Critical|High|Medium|Low",
        "actions": "list of all actions to be taken",
        "primary_actions": [
            {{
            "action": "description of action",
            "party": "<actual party name from party_contributions>",
            "network_tier": "RAN|Edge|Core",
            "party_evidence_type": "<party name or evidence type>",
            "rationale": "why this party should execute this action"
            }}
        ],
        "supporting_actions": [
            {{
            "action": "description of action",
            "party": "<actual party name from party_contributions>",
            "network_tier": "RAN|Edge|Core",
            "party_evidence_type": "<party name or evidence type>",
            "rationale": "why this party should execute this action"
            }}
        ],
        "reasoning": "Brief explanation of the decision and party assignments",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": ["source1", "source2"]
        }}

        IMPORTANT: 
        - Each action in primary_actions and supporting_actions must be an object with "action", "party", "network_tier", "party_evidence_type", and "rationale" fields.
        - The "party" field must use the actual party name from the party_contributions (e.g., "RAN_device_party1", "Edge_device_party2", "Core_device_party3").
        - The "network_tier" field must be "RAN", "Edge", or "Core" based on the party's network tier.
        - The "party_evidence_type" should describe the type of evidence this party provides.
        - Actions should be assigned to parties based on which network tier and evidence type is most relevant to that action.
        - Consider the dominant network tier ({dominant_tier if dominant_tier else "N/A"}) when prioritizing actions.
        """

    return prompt
    


In [18]:
def create_prompt_without_RAG(sample_data):
    """Create prompt with prediction details, party evidence, contributing features, WITHOUT RAG context."""
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    

    
    prompt = f"""You are a cybersecurity expert analyzing attack predictions and generating action plans.
      The goal is to support agent decisions on WHICH evidence party should trigger actions
      and WHAT actions are appropriate based on domain knowledge.

      Prediction summary:
      - sample_id: {sample_id}
      - predicted_label: {predicted_label}
      - confidence: {confidence:.1%}

      Output:
      Return a JSON object with the following structure:
      {{
        "threat_level": "Critical|High|Medium|Low",
        "actions:"List of all actions",
        "reasoning": "Brief explanation of the decision and party assignments",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": ["source1", "source2", ...]
      }}"""

   
    return prompt

In [19]:
# 3. RAG Search Function
def search_rag_knowledge(query, top_k=5, prioritize_agentic_features=True):
    """Search knowledge base and return documents with title and text.
    
    Args:
        query: Search query string
        top_k: Number of results to return
        prioritize_agentic_features: If True, prioritize documents from agentic_features_actions.json
    
    Returns:
        List of formatted results with title, text, and similarity score
    """
    if vector_store is None:
        return []
    
    # Get more results initially to allow for prioritization
    search_k = top_k * 2 if prioritize_agentic_features else top_k
    results = vector_store.similarity_search_with_score(query, k=search_k)
    
    formatted_results = []
    agentic_results = []  # Results from agentic_features_actions.json
    other_results = []  # Other results
    seen_titles = set()
    
    for doc, score in results:
        title = doc.metadata.get("title", "Unknown")
        text = doc.metadata.get("text", "")
        
        # Avoid duplicates - use first occurrence of each title
        if title not in seen_titles:
            seen_titles.add(title)
            similarity_score = 1 / (1 + score) if score > 0 else 1.0
            
            result_item = {
                "title": title,
                "text": text,
                "score": similarity_score
            }
            
            # Prioritize documents from agentic_features_actions.json
            # Check if title contains agentic features keywords
            if prioritize_agentic_features and (
                "Agentic Features" in title or 
                "RAN" in title or 
                "Edge Network" in title or 
                "Core Network" in title or
                "Features and Actions" in title
            ):
                agentic_results.append(result_item)
            else:
                other_results.append(result_item)
    
    # Combine results: agentic_features first, then others
    if prioritize_agentic_features:
        # Take top results from agentic_features, then fill remaining slots with others
        formatted_results = agentic_results[:top_k]
        remaining_slots = top_k - len(formatted_results)
        if remaining_slots > 0:
            formatted_results.extend(other_results[:remaining_slots])
    else:
        formatted_results = (agentic_results + other_results)[:top_k]
    
    return formatted_results

In [20]:
# 8. Save Comparison File Function
def save_comparison_file(sample, query, rag_results, llm_response_with_rag, llm_response_without_rag, 
                         results_action_dir, with_rag_file, without_rag_file):
    """Save a comparison JSON file containing both with-RAG and without-RAG action plans.
    
    Args:
        sample: Prediction sample data
        query: RAG query used
        rag_results: RAG search results
        llm_response_with_rag: LLM response with RAG context
        llm_response_without_rag: LLM response without RAG context
        results_action_dir: Directory to save results
        with_rag_file: Path to the with-RAG action plan file
        without_rag_file: Path to the without-RAG action plan file
    """
    comparison = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0))
        },
        "comparison_metadata": {
            "compared_at": datetime.now().isoformat(),
            "with_rag_file": with_rag_file.name if with_rag_file else None,
            "without_rag_file": without_rag_file.name if without_rag_file else None,
            "rag_query": query,
            "rag_documents_found": len(rag_results)
        },
        "rag_context": {
            "documents_used": len(rag_results),
            "top_documents": [
                {
                    "title": r["title"],
                    "similarity_score": float(r["score"]),
                    "text_preview": r["text"][:200] + "..." if len(r["text"]) > 200 else r["text"]
                }
                for r in rag_results[:3]
            ]
        },
        "action_plans": {
            "with_rag": {
                "threat_level": llm_response_with_rag.get("threat_level") if llm_response_with_rag else None,
                "execution_priority": llm_response_with_rag.get("execution_priority") if llm_response_with_rag else None,
                "primary_actions": llm_response_with_rag.get("primary_actions", []) if llm_response_with_rag else [],
                "supporting_actions": llm_response_with_rag.get("supporting_actions", []) if llm_response_with_rag else [],
                "reasoning": llm_response_with_rag.get("reasoning", "") if llm_response_with_rag else "",
                "knowledge_sources_used": llm_response_with_rag.get("knowledge_sources_used", []) if llm_response_with_rag else [],
            },
            "without_rag": {
                "threat_level": llm_response_without_rag.get("threat_level") if llm_response_without_rag else None,
                "execution_priority": llm_response_without_rag.get("execution_priority") if llm_response_without_rag else None,
                "primary_actions": llm_response_without_rag.get("primary_actions", []) if llm_response_without_rag else [],
                "supporting_actions": llm_response_without_rag.get("supporting_actions", []) if llm_response_without_rag else [],
                "reasoning": llm_response_without_rag.get("reasoning", "") if llm_response_without_rag else "",
                "knowledge_sources_used": llm_response_without_rag.get("knowledge_sources_used", []) if llm_response_without_rag else [],
            }
        },
        "differences": {
            "threat_level_different": (llm_response_with_rag.get("threat_level") if llm_response_with_rag else None) != 
                                      (llm_response_without_rag.get("threat_level") if llm_response_without_rag else None),
            "priority_different": (llm_response_with_rag.get("execution_priority") if llm_response_with_rag else None) != 
                                 (llm_response_without_rag.get("execution_priority") if llm_response_without_rag else None),
            "primary_actions_different": (llm_response_with_rag.get("primary_actions", []) if llm_response_with_rag else []) != 
                                        (llm_response_without_rag.get("primary_actions", []) if llm_response_without_rag else []),
            "num_primary_actions_with_rag": len(llm_response_with_rag.get("primary_actions", [])) if llm_response_with_rag else 0,
            "num_primary_actions_without_rag": len(llm_response_without_rag.get("primary_actions", [])) if llm_response_without_rag else 0
        }
    }
    
    # Convert any numpy types to native Python types
    comparison = convert_to_json_serializable(comparison)
    
    # Create filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparison_file = results_action_dir / f"comparison_sample_{sample.get('sample_id')}_{timestamp}.json"
    
    with open(comparison_file, "w", encoding="utf-8") as f:
        json.dump(comparison, f, indent=2, ensure_ascii=False)
    
    return comparison_file

In [21]:
# 7. Save Action Plan
def convert_to_json_serializable(obj):
    """Convert numpy types and other non-serializable types to JSON-compatible types."""
    import numpy as np
    
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_json_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_json_serializable(item) for item in obj]
    else:
        return obj

def save_action_plan(sample, query, rag_results, llm_response, results_action_dir):
    """Save action plan in the specified format."""
    result = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0))
        },
        "rag_info": {
            "documents_used": len(rag_results),
            "search_method": "vector_similarity",
            "search_queries": [query] if query else [],
            "top_results": [
                {
                    "title": r["title"],
                    "text": r["text"],
                    "similarity_score": float(r["score"]),
                    "source_file": str(rag_documents)
                }
                for r in rag_results[:3]
            ]
        },
        "llm_action_plan": llm_response,
        "processed_at": datetime.now().isoformat()
    }
    
    # Convert any numpy types to native Python types
    result = convert_to_json_serializable(result)
    
    # Create filename with timestamp (include microseconds to avoid collisions)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_file = results_action_dir / f"action_plan_sample_{sample.get('sample_id')}_{timestamp}.json"
    
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        # Verify file was created
        if not output_file.exists():
            raise FileNotFoundError(f"File was not created: {output_file}")
        return output_file
    except Exception as e:
        print(f"Error in save_action_plan: {e}")
        raise

In [22]:
# 6. Call LLM API
def call_llm_api(prompt):
    """Call LLM API and return JSON response."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None
    
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a cybersecurity expert. Return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find('{')
        end = response_text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")
    
    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": []
    }

# 7. Process Predictions
# Process ALL predictions (including BENIGN) with RAG and attack-focused prompts
for sample in predictions_data:
    print("="*80)
    print(f"Processing Sample ID: {sample.get('sample_id')}")
    predicted_label = sample.get('predicted_label', 'UNKNOWN')
    print(f"Predicted Label: {predicted_label}")
    print(f"Confidence: {sample.get('confidence', 0):.1%}")
    
    # Process ALL predictions with RAG (including BENIGN)
    # Create smart RAG query from contributing features
    query = build_dynamic_rag_query(sample)
    print(f"\nRAG Query: {query}")
    
    # Search knowledge base (get top 5, use top 3)
    rag_results = search_rag_knowledge(query, top_k=5)
    print(f"Found {len(rag_results)} relevant documents")
    
    if rag_results:
        print("\nTop 3 documents for LLM context:")
        for idx, result in enumerate(rag_results[:3], 1):
            print(f"  {idx}. {result['title']} (Score: {result['score']:.2%})")
    
    # Create prompt with top 3 RAG results (for all predictions including BENIGN)
    prompt = create_prompt(sample, rag_results)
    prompt_without_rag = create_prompt_without_RAG(sample)
    
    # Call LLM with RAG
    print("\nCalling LLM API with RAG...")
    llm_response_with_rag = call_llm_api(prompt)
    
    # Call LLM without RAG
    print("\nCalling LLM API without RAG...")
    llm_response_without_rag = call_llm_api(prompt_without_rag)
    
    # Save with-RAG result
    output_file_with_rag = None
    if llm_response_with_rag:
        print(f"\n[With RAG] Threat Level: {llm_response_with_rag.get('threat_level')}")
        print(f"[With RAG] Priority: {llm_response_with_rag.get('execution_priority')}")
        
        # Save result using reusable function
        try:
            output_file_with_rag = save_action_plan(sample, query, rag_results, llm_response_with_rag, results_action_dir)
            print(f"✓ Saved with-RAG result to {output_file_with_rag.name}")
            print(f"  Full path: {output_file_with_rag}")
        except Exception as e:
            print(f"✗ Error saving with-RAG result: {e}")
            output_file_with_rag = None
    else:
        print("✗ Failed to get LLM response with RAG")
        output_file_with_rag = None
    
    # Save without-RAG result
    without_rag_path = None
    if llm_response_without_rag:
        print(f"\n[Without RAG] Threat Level: {llm_response_without_rag.get('threat_level')}")
        print(f"[Without RAG] Priority: {llm_response_without_rag.get('execution_priority')}")
        
        # Save without-RAG result (pass empty rag_results)
        output_file_without_rag = save_action_plan(sample, query, [], llm_response_without_rag, results_action_dir)
        # Rename to indicate it's without RAG
        without_rag_path = output_file_without_rag.parent / output_file_without_rag.name.replace("action_plan_", "action_plan_noRAG_")
        output_file_without_rag.rename(without_rag_path)
        print(f"✓ Saved without-RAG result to {without_rag_path.name}")
    else:
        print("✗ Failed to get LLM response without RAG")
        output_file_without_rag = None
    
    # Save comparison file if both responses exist
    if llm_response_with_rag and llm_response_without_rag:
        comparison_file = save_comparison_file(
            sample, query, rag_results, 
            llm_response_with_rag, llm_response_without_rag,
            results_action_dir, output_file_with_rag, without_rag_path
        )
        print(f"✓ Saved comparison file to {comparison_file.name}")
    
    print("="*80)

Processing Sample ID: 0
Predicted Label: BENIGN
Confidence: 87.1%

RAG Query: What mitigation actions are required for BENIGN attack detected in Edge network tier (Edge contributing 51.3%) with elevated UDP traffic patterns with contributing features (>0%): udps.srcdst_rst_packet_count, udps.srcdst_tcp_packet_count, bidirectional_duration_ms, udps.udp_packet_count, udps.srcdst_psh_packet_count, udps.ack_packet_count, udps.bidirectional_packet_count, udps.srcdst_fin_packet_count, udps.srcdst_src2dst_packet_count, bidirectional_min_ps, bidirectional_packets, bidirectional_ece_packets, bidirectional_syn_packets, dst2src_max_piat_ms, bidirectional_max_ps, dst2src_fin_packets, udps.bidirectional_duration_avg based on agentic feature contributions and network tier analysis.
Found 5 relevant documents

Top 3 documents for LLM context:
  1. Edge Network Features and Actions (Score: 53.93%)
  2. Agentic Features and Actions - Overview (Score: 51.55%)
  3. What are the key differences between ne